# D2.1 · How do we know a model learned?

**PowerPoint slides** 3–8 and 13  
**Course data** property_valuations.csv

### The problem

If we know how large a Dubai apartment is, can we guess its value? Which way of guessing works best on later examples?

### What this demo shows

Compare a middle-value guess, a straight line and three trees. A tree asks a series of questions such as whether the apartment is larger than a certain size. Depth tells us how many levels of questions the tree may ask.

### Why we’re doing it

We need to see how each method handles apartments it has never seen. That is why we keep a separate group for choosing a method and another for one final check.

**If this is your first ML activity**, open [00 · Your first ML workflow](00_Start_Here_Your_First_ML_Workflow.ipynb) first. For the simpler examples of model types and repeated checking, see [01 · Regression and classification](01_Regression_and_Classification.ipynb) and [03 · K-fold cross-validation](03_K_Fold_Cross_Validation.ipynb).

Here, read the plain-English explanation before each cell. Run **Setup**, then work down the page with **Shift + Enter**.

## Setup · Load the data

Read six columns, keep Dubai apartments and take a fixed sample from each time period. The table shows three training examples. The first few lines locate the course data folder. You can
run the whole cell as written.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor

working_folder = Path.cwd()
data_folder = next(
    (folder / 'data' for folder in (working_folder, *working_folder.parents)
     if (folder / 'data' / 'property_valuations.csv').is_file()),
    None,
)
if data_folder is None:
    raise FileNotFoundError('Open the extracted course folder in Jupyter or VS Code.')

columns = ["valuation_date", "emirate", "property_type", "size_sqft", "bedrooms", "valuation_aed"]
properties = pd.read_csv(data_folder / "property_valuations.csv", usecols=columns, parse_dates=["valuation_date"])
apartments = properties.query("emirate == 'Dubai' and property_type == 'Apartment'")
apartments = apartments.query("size_sqft > 0 and valuation_aed > 0").dropna()
train = apartments[apartments["valuation_date"] < "2024-01-01"].sample(1000, random_state=42)
validation = apartments[apartments["valuation_date"].dt.year == 2024].sample(500, random_state=42)
test = apartments[apartments["valuation_date"].dt.year == 2025].sample(500, random_state=42)
display(train[["size_sqft", "bedrooms", "valuation_aed"]].head(3))

## 1 · Fit, choose, then check

**What we’re doing.** Print the years and counts for the three groups of rows.

**Why we’re doing it.** Older examples teach the model. The next year helps us choose a method. The latest year gives a fresh check after we have chosen.

**What to look for.** The three periods run in order. The sample uses 1,000 training rows and 500 rows in each later set.

In [ ]:
for name, rows in [("Train", train), ("Validation", validation), ("Test", test)]:
    dates = rows["valuation_date"]
    print(name, "·", len(rows), "rows ·", dates.min().date(), "to", dates.max().date())

## 2 · Start with a simple guess

**What we’re doing.** Use the middle value of the training apartments as everyone’s starting guess.

**Why we’re doing it.** We can make this guess quickly. Any method we teach to a computer should be checked against a simple starting point, called a baseline.

**What to look for.** The median guess is about AED 932,000 and the validation error is about AED 822,000.

In [ ]:
baseline = train["valuation_aed"].median()
baseline_error = (validation["valuation_aed"] - baseline).abs().mean()
print(f"Median guess: AED {baseline:,.0f}")
print(f"Validation MAE: AED {baseline_error:,.0f}")

## 3 · Compare a line and three trees

**What we’re doing.** Compare the misses from a straight line and three trees on both training and validation rows.

**Why we’re doing it.** A deep tree can learn rules that suit individual training examples. Validation shows whether those rules work for unfamiliar apartments. A model that is too simple can miss the pattern on both groups.

**What to look for.** The deep tree has the lowest training error. The straight line has the lowest validation error in this sample.

In [ ]:
inputs = ["size_sqft"]
target = "valuation_aed"
models = {
    "Straight line": LinearRegression(),
    "Depth 1": DecisionTreeRegressor(max_depth=1, random_state=42),
    "Depth 4": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Deep tree": DecisionTreeRegressor(random_state=42),
}
errors = []
for name, model in models.items():
    model.fit(train[inputs], train[target])
    train_error = mean_absolute_error(train[target], model.predict(train[inputs]))
    validation_error = mean_absolute_error(validation[target], model.predict(validation[inputs]))
    errors.append([name, train_error, validation_error])
results = pd.DataFrame(errors, columns=["Model", "Train MAE", "Validation MAE"])
display(results.round(0))

### See the training and validation misses side by side

For each model, compare the first bar with the second. The deep tree has a small training bar and a much taller validation bar.

In [ ]:
import matplotlib.pyplot as plt

chart = results.set_index("Model")[["Train MAE", "Validation MAE"]] / 1000
ax = chart.plot.bar(figsize=(8, 3.4), color=["#2676a8", "#c67718"], rot=0)
ax.set(title="How large were the misses?", xlabel="", ylabel="Average miss (thousand AED)")
plt.tight_layout()
plt.show()

## 4 · Try more training examples

**What we’re doing.** Use the same straight line with 40, 200 and then 1,000 examples. Check against the same 500 apartments each time.

**Why we’re doing it.** Keeping the check group fixed lets us see what changed when the model had more examples to learn from.

**What to look for.** Compare the three errors. Small samples can give jumpier results. Adding rows can help without guaranteeing improvement on every run.

In [ ]:
for count in [40, 200, 1000]:
    small = train.head(count)
    line = LinearRegression().fit(small[inputs], small[target])
    error = mean_absolute_error(validation[target], line.predict(validation[inputs]))
    print(f"{count:>4} training rows → validation MAE AED {error:,.0f}")

## 5 · See how validation moves

**What we’re doing.** Print several ways to use an earlier period for training and a later year for validation.

**Why we’re doing it.** A single year can be unusually easy or hard to predict. Several checks show whether the result holds up, while keeping future examples out of training.

**What to look for.** The training period grows and the validation year moves forward. 2025 remains outside these splits.

In [ ]:
years = list(range(2018, 2025))
for earlier, later in TimeSeriesSplit(n_splits=3).split(years):
    print("Train through", years[earlier[-1]], "→ validate on", years[later[0]])

| Split | When it helps |
|---|---|
| K-fold | Rotate the held-out block on rows without a required time order. Summarise the fold scores with a mean and spread. |
| Stratified K-fold | Keep approximately the same class proportions in each fold. |
| Group K-fold | Keep all rows for a person or another group together. |
| TimeSeriesSplit | Train on earlier observations and validate on later ones. |
| Nested cross-validation | Use inner folds to choose settings and outer folds to check that selection process. |

## 6 · Read a learned value and a setting

**What we’re doing.** Read the slope of the line and the depth of one tree.

**Why we’re doing it.** The line learns its slope from the data. We choose the depth before we fit a tree. This difference tells us which values we can try changing.

**What to look for.** The slope has units and comes from the data. Depth 4 was a choice in the model setup.

In [ ]:
slope = models["Straight line"].coef_[0]
print(f"Learned slope: AED {slope:,.0f} per extra square foot")
print("Chosen maximum tree depth:", models["Depth 4"].max_depth)

## 7 · Take a few learning steps

**What we’re doing.** Show what happens when the computer changes the slope by small, medium or large amounts.

**Why we’re doing it.** The number called loss measures the size of the mistakes. We want updates that make it smaller. Very large updates can jump past a better answer.

**What to look for.** The moderate rate reduces the loss quickly in this example. With the oversized rate, the losses rise. Each update here uses the full training batch.

In [ ]:
size = train["size_sqft"] / 1000
value = train["valuation_aed"] / 1_000_000
x = (size - size.mean()) / size.std()
y = value - value.mean()
for rate in [0.01, 0.3, 1.2]:
    slope_guess = 0.0
    losses = []
    for update in range(6):
        error = slope_guess * x - y
        gradient = 2 * (error * x).mean()
        slope_guess = slope_guess - rate * gradient
        loss = ((slope_guess * x - y) ** 2).mean()
        losses.append(round(float(loss), 3))
    print("Rate", rate, "→ loss after each update", losses)

**Read the setup aloud.** We centre the values and scale the input to make
the updates easier to compare. Full-batch uses every training row at each
update. A mini-batch uses a small group. Stochastic gradient descent can
use one row at a time. For a straight line, ordinary least squares can also
solve for its coefficients directly.

## 8 · Open the final test set

**What we’re doing.** Pick the model with the smallest validation miss. Use it to guess values for the 2025 apartments.

**Why we’re doing it.** The 2025 answers stayed aside while we chose the model, so this gives us a fresh check.

**What to look for.** The selected straight line shows an MAE of about AED 703,000 on this sample.

In [ ]:
chosen_name = results.sort_values("Validation MAE").iloc[0]["Model"]
chosen_model = models[chosen_name]
final_error = mean_absolute_error(test[target], chosen_model.predict(test[inputs]))
print(f"{chosen_name} · final 2025 MAE AED {final_error:,.0f}")

### Your turn

Before opening the final test cell, predict which validation model you would
select. Point to the number that supports your choice.